# Decoding Algorithms

Run the setup cell once, then run each decoding cell. At the bottom of the notebook is a Playground to change the parameters of the decoding algorithms we covered in class.

In [ ]:
%pip -q install transformers torch pandas

import torch
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# Pick a model, then rerun this cell:
MODEL = "distilbert/distilgpt2"       # fastest
# MODEL = "openai-community/gpt2"     # larger GPT-2
# MODEL = "EleutherAI/pythia-160m"    # small alternative model

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

PROMPT = "Once upon a time, there was a cat named Charlie "
MAX_NEW_TOKENS = 30
SEED = 42

def generate(
    method,
    prompt=None,
    beam_size=5,
    k=40,
    p=0.9,
    temperature=1.0,
    max_new_tokens=None,
    seed=SEED,
    length_penalty=1.0,
    return_ids=False,
):
    method = method.lower()
    prompt = PROMPT if prompt is None else prompt
    max_new_tokens = MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens

    if seed is not None:
        set_seed(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    args = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
    }

    if method == "greedy":
        args.update(do_sample=False)

    elif method == "beam":
        if beam_size < 2:
            raise ValueError("beam_size must be at least 2 for beam search.")
        args.update(
            do_sample=False,
            num_beams=beam_size,
            length_penalty=length_penalty,
        )

    elif method == "sample":
        if temperature <= 0:
            raise ValueError("temperature must be greater than 0.")
        args.update(
            do_sample=True,
            top_k=0,
            top_p=1.0,
            temperature=temperature,
        )

    elif method == "top-k":
        if k < 1:
            raise ValueError("k must be at least 1.")
        if temperature <= 0:
            raise ValueError("temperature must be greater than 0.")
        args.update(
            do_sample=True,
            top_k=k,
            top_p=1.0,
            temperature=temperature,
        )

    elif method == "top-p":
        if not 0 < p <= 1:
            raise ValueError("p must be between 0 and 1.")
        if temperature <= 0:
            raise ValueError("temperature must be greater than 0.")
        args.update(
            do_sample=True,
            top_k=0,
            top_p=p,
            temperature=temperature,
        )

    else:
        raise ValueError(
            'method must be "greedy", "beam", "sample", "top-k", or "top-p".'
        )

    output = model.generate(**inputs, **args)[0]

    if return_ids:
        return output
    return tokenizer.decode(output, skip_special_tokens=True)


def next_token_probs(input_ids, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0.")

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1] / temperature

    return torch.softmax(logits, dim=-1)


def show_top10(generated, prompt, method, temperature=1.0):
    """Show top-10 next-token probabilities along this exact generation."""
    method = method.lower()

    # Greedy and beam search do not use temperature.
    distribution_temperature = (
        temperature if method in {"sample", "top-k", "top-p"} else 1.0
    )

    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)[0]
    prompt_len = len(prompt_ids)
    rows = []

    for step, chosen_id in enumerate(generated[prompt_len:]):
        prefix = generated[: prompt_len + step].unsqueeze(0)
        probs = next_token_probs(prefix, distribution_temperature)
        top_probs, top_ids = torch.topk(probs, 10)

        row = {
            "step": step + 1,
            "chosen": repr(tokenizer.decode([chosen_id.item()])),
        }

        for i, (token_id, prob) in enumerate(zip(top_ids, top_probs), start=1):
            row[f"#{i}"] = (
                f"{tokenizer.decode([token_id.item()])!r} ({prob.item():.3f})"
            )

        rows.append(row)

    display(pd.DataFrame(rows))


## Greedy decoding

**Strength:** fast and deterministic.  
**Limitation:** locally best choices can lead to worse or repetitive sequences.

In [ ]:
print(generate("greedy"))

## Beam search

**Strength:** keeps several candidate sequences instead of only one.  
**Limitation:** costs more as beam size grows, and larger beams do not guarantee better text.

`LENGTH_PENALTY = 0.0` uses the unnormalized accumulated log-probability.  
`LENGTH_PENALTY = 1.0` divides the score by sequence length.

Length penalty only changes the ranking when candidate sequences have different lengths.

In [ ]:
BEAM_SIZE = 5
LENGTH_PENALTY = 1.0   # 0.0 = no length normalization

print(
    generate(
        "beam",
        beam_size=BEAM_SIZE,
        length_penalty=LENGTH_PENALTY,
    )
)

## Pure sampling

**Strength:** produces diverse outputs.  
**Limitation:** low-probability tokens in the tail can still be sampled.

In [ ]:
TEMPERATURE = 1.0
print(generate("sample", temperature=TEMPERATURE))

## Top-\(K\) sampling

**Strength:** removes the low-probability tail.  
**Limitation:** a fixed \(K\) does not adapt to how peaked or flat the distribution is.

In [ ]:
K = 40
TEMPERATURE = 1.0
print(generate("top-k", k=K, temperature=TEMPERATURE))

## Top-\(P\) / nucleus sampling

**Strength:** adapts the number of candidate tokens to the shape of the distribution.  
**Limitation:** \(P\) still needs to be chosen: small \(P\) can be too conservative, while large \(P\) can include unlikely tokens.

In [ ]:
P = 0.50
TEMPERATURE = 1.0
print(generate("top-p", p=P, temperature=TEMPERATURE))

## Playground

Change this cell to compare decoding methods and parameters.

Set `SEED` to an integer if you want reproducible sampling. Otherwise leave it as `None`.

Set `SHOW_TOP10 = True` to see the top 10 next-token probabilities at every time step. For top-\(K\) and top-\(P\), these are shown **before** truncation.

In [ ]:
PROMPT = "Once upon a time, there was a cat named Charlie "

METHOD = "top-p"       # "greedy", "beam", "sample", "top-k", or "top-p"
BEAM_SIZE = 5
LENGTH_PENALTY = 1.0   # beam only; 0.0 = unnormalized
K = 40
P = 0.50
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 30

SEED = 42           # None = new samples; integer = repeatable samples
SHOW_TOP10 = True     # True = inspect the distribution at every step

generated = generate(
    METHOD,
    prompt=PROMPT,
    beam_size=BEAM_SIZE,
    length_penalty=LENGTH_PENALTY,
    k=K,
    p=P,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
    return_ids=True,
)

print(tokenizer.decode(generated, skip_special_tokens=True))

if SHOW_TOP10:
    show_top10(generated, PROMPT, METHOD, TEMPERATURE)
